link da documentação da API do Youtube com o passo a passo da utilização
da API: https://developers.google.com/youtube/v3/quickstart/python?hl=pt-br

Observação: Será necessário você adicionar o seu email como usuário de teste no seu projeto da google cloud

credenciais.json: Arquivo json que possui as credenciais necessárias para o script acessar o sistema. Sem ela, os mecanismos de defesa do google não possibilitam o acesso a API do seu projeto.

In [ ]:
# Biblioteca que possibilita a xomunicação do python com o sistema de
# arquivos do sistema operacional
import os

# Esta biblioteca serve especificamente para resolver o protocolo OAuth 2.0,
# que é o sistema de segurança que o Google usa para permitir que seu
# script acesse dados. Ela é a responsável por abrir a janela do navegador
# web, pedir para você fazer login com a sua conta do Gmail, mostrar a tela
# de permissões e, após a sua autorização, capturar o token de acesso (a
# chave de segurança).
import google_auth_oauthlib.flow

# Esta é a bilioteca principal que simplifica a nossa vida para conversar
# com os servidores do Google. O termo discovery (descoberta) em tecnologia
# se refere a uma técnica onde a biblioteca baixa dinamicamente um mapa
# de tudo o que a API do youtube sabe fazer. Ela pega as suas credenciais
# validadas e constroi um "objeto cliente". A partir desse objeto, você 
# ganha acesso a todos os comandos do YouTube sem precisar escrever 
# requisições HTTP manuais.
import googleapiclient.discovery

# Link da requisição a API que possui as permissões que o usuário de teste
# pode ter no sistema. Nesse caso, o nosso usuário so tem permissão para ler
# os dados (readonly)
scopes = ["https://www.googleapis.com/auth/youtube.readonly"]

# Função que irá conter todo a requisição e coleta das informações dos videos
def main():
    
    # Por padrão, o protocolo de segurança do Google exige conexões 
    # criptografadas (https) para trafegar dados de login. Como estamos
    # desenvolvendo e testando o script de forma local no nosso computador
    # (localhost), a comunicação ocorre em HTTP simples. Essa linha altera
    # uma variável no ambiente do sistema para avisar a biblioteca de
    # autenticação: "pode permitir conexões locais não criptografadas,
    # estamos apenas em ambiente de teste". Sem isso, o Pythob travaria
    # o programa por segurança
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

    # Variaveis de identificação: Criam strings constantes que definem o alvo
    # do seu script: o nome do serviço (api_service_name), a versão atual da
    # API (api_version) e o caminho para o arquivo que contém as suas chaves
    # de acesso (client_secrets_file).
    api_service_name = "youtube"
    api_version = "v3"
    client_secrets_file = "credenciais.json"

    # Essa função carrega o arquivo credenciais.json e lê as chaves do seu
    # projeto (Client ID e Client Secret), além de anexar a lista de escopos
    # (scopes) que diz ao Google qual o nivel de acesso que o nosso código
    # está solicitando.
    flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
        client_secrets_file, scopes)
    
    # O run_local_server: É a linha que abre a janela do seu navegador de 
    # internet. Ela cria um servidor local temporário na nossa máquina
    # para interceptar a resposta do Google. Quando clicamos em "permitir"
    # na página do Google, esse servidor captura o token de acesso gerado
    # e o armazena na variável credentials.
    credentials = flow.run_local_server()
    
    # Esta função pega esse token armazenado, valida-o com os servidores
    # do Google e constrói dinamicamente um objeto cliente (salvo na variável
    # youtube). A partir daqui, esse objeto possui todos os métodos e comandos
    # mapeados da API do YouTube.
    youtube = googleapiclient.discovery.build(
        api_service_name, api_version, credentials=credentials)

    # Prepara um pedido para listar as informações dos videos de dentro
    # de uma playlist especifica. Note os parâmetros:
    request = youtube.playlistItems().list(
         # Diz a API para trazer apenas os detalhes de
         # conteúdo (onde fica armazenado o ID do video). Pedir apenas o que
         # vai usar poupa o tempo de resposta e consome menos cota da API.
        part="contentDetails",
        
        # Limita o retorno a no máximo 25 videos.
        maxResults=25,
        
         # O ID da playlist que configuramos.
        playlistId="PLMjbgjzww2EH3aU9qaF_OPFBWrpJayzAH"
    )
    
    # Faz a chamada de rede real via internet. O resultado retornado do servidor
    # do Google vem no formato JSON e é salvo na variável response como um
    # dicionário do Python.
    response = request.execute()
    
    # Ira coletar apenas a lista de videos que veio dentro da chave "items"
    videos = response["items"]
    
    # Ira imprimir o dicionário json com as informações dos videos presentes
    # na playlist escolhida.
    print(response)
    
    # Uma lista comprehension que varre cada item retornado e navega pelas
    # chaves aninhadas(['contentDetails']['videoID]) para extrair apenas o
    # texto com o ID do video, gerando uma lista limpa de strings (ex: ['ID_VIDEO1', 'ID_VIDEO2'])
    ids_video = [video["contentDetails"]["videoId"] for video in videos]
    
    # O endpoint de playlist não possui dados de engajamento (como visualizações)
    # para obte-los, você precisa consultar a coleção de videos diretamente.
    request = youtube.videos().list(
        # Solicita especificamente o bloco de estatisticas (contagem de views,
        # likes, comentários).
        part="statistics",
        
        # O método ",".join() junta toda a sua lista de IDs em uma única 
        # string separada por virgulas (ex: "ID1, ID2, ID3"). A API do
        # youtube permite passar multiplos IDs de uma vez, o que significa
        # que você faz uma única requisição para coletar dados de 25 videos,
        # em vez de fazer 25 requisições individuais.
        id=",".join(ids_video)
    )
    
    # Envia esse lote de IDs ao Google e salva a resposta estruturada em response.
    response = request.execute()
    videos = response["items"]   
    
    # O loop passa por cada video retornado na segunda consulta, extrai o 
    # identificador único video (video["id"]), o número total de visualizações
    # acumuladas até o momento (video["statistics"]["viewCount"]) e exibe
    # os dois formatados no seu terminal com o print.
    for video in videos:
        id_video = video["id"]
        views = video["statistics"]["viewCount"]
        print(id_video, "-", views)

# É uma estrutura padrão em Python para garantir boas práticas. Ela diz que
# a função main() só será disparada se você executar esse arquivo diretamente
# (rodando o script ou a célula do Jupyter). Se um dia você importar esse
#  arquivo como um módulo dentro de outro script, o código não saira rodando
# sozinho sem o seu comando.
if __name__ == "__main__":
    main()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=549856275678-927936p0egffpos46kpcmmhu7c56m21a.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.readonly&state=JQHIgttZwHYZQ6CjQvZUJkTMD8Aw2r&code_challenge=70wDA5xFVQqoiy_V-dkvs-j_KOO7zIVVkp3RoLsVtNI&code_challenge_method=S256&access_type=offline
{'kind': 'youtube#playlistItemListResponse', 'etag': 'wy2Ra-AImWnpGyKTV_HZz5RmPMM', 'nextPageToken': 'EAAaHlBUOkNCa2lFRGN4TWpVME1qQTVNekJDTWpFek0wWQ', 'items': [{'kind': 'youtube#playlistItem', 'etag': 'erpQbpfmW3vGCUrVUJAzne0G4Rc', 'id': 'UExNamJnanp3dzJFSDNhVTlxYUZfT1BGQldycEpheXpBSC41NkI0NEY2RDEwNTU3Q0M2', 'contentDetails': {'videoId': 'FeB4f0CK1Ek', 'videoPublishedAt': '2014-07-29T03:34:43Z'}}, {'kind': 'youtube#playlistItem', 'etag': 'SZIv5pl5LLcAtmIdELIszf_kDLw', 'id': 'UExNamJnanp3dzJFSDNhVTlxYUZfT1BGQldycEpheXpBSC4yODlGNEE0NkRGMEEzMEQy